In [ ]:
import math
import random
import json
import os
import asyncio
import pandas as pd
from tqdm.notebook import tqdm
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_string, encode_image
from vpei.common_variables import *
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_utils import (
    print_comparative_experiment_results,
    generate_n_different_full_names_initials,
    build_user_prompt_template_with_variable_repeats,
)
from vpei.epistemic_consistency.art.local_variables import PAINTING_STYLES

In [ ]:
experiment_name = "art"
experiment_type = "comparative_experiment_without_ground_truth_and_multiple_choices"
system_prompt = EXPERIMENTS[experiment_name][experiment_type]["system_prompt"]
user_prompt_template_repeated_block = EXPERIMENTS[experiment_name][experiment_type]["user_prompt_template_repeated_block"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template_repeated_block)

In [ ]:
async def _wrap_awaitable(index, awaitable):
    try:
        result = await awaitable
        return index, result
    except Exception as e:
        print(f"Error in task {index}: {e}")
        return index, None


async def carry_out_art_comparative_experiment_without_ground_truth_and_multiple_choices(
    models, n, system_prompt, user_prompt_template_repeated_block, data_path, number_of_choices=5,
    custom_model_kwargs={}, path_to_save_model_outputs="./comparative_experiment_without_ground_truth_and_multiple_choices",
    random_seed=42, **kwargs
):
    _POLITICAL_ATTITUDES_CATEGORIES = kwargs.get("POLITICAL_ATTITUDES_CATEGORIES", POLITICAL_ATTITUDES_CATEGORIES)
    user_prompt_template = build_user_prompt_template_with_variable_repeats(
        number_of_choices, user_prompt_template_repeated_block
    )

    # Collect all available paintings
    all_paintings = []
    for painting_style in PAINTING_STYLES:
        style_path = f"{data_path}/{painting_style}"
        if not os.path.exists(style_path):
            continue
        available_images = [f for f in os.listdir(style_path) if f.endswith(".png")]
        for img in available_images:
            all_paintings.append((painting_style, img))

    async def run_model(model_name, position=0):
        random.seed(random_seed)
        model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs=custom_model_kwargs)
        tasks = []

        for i in range(n):
            if len(all_paintings) < number_of_choices:
                print("Not enough paintings")
                continue

            selected_paintings = random.sample(all_paintings, number_of_choices)
            names = generate_n_different_full_names_initials(number_of_choices)

            # Assign political attitudes across categories
            n_categories = math.ceil(number_of_choices / 2)
            political_attitude_categories = random.sample(list(_POLITICAL_ATTITUDES_CATEGORIES.keys()), n_categories)
            political_attitudes = {}
            political_poles = {}
            count = 0
            for cat in political_attitude_categories:
                for pole in random.sample(["right", "left"], k=2):
                    if count >= number_of_choices:
                        break
                    idx = count + 1
                    political_attitudes[f"political_attitude_{idx}"] = _POLITICAL_ATTITUDES_CATEGORIES[cat][pole]
                    political_poles[f"political_pole_{idx}"] = pole
                    count += 1
                if count >= number_of_choices:
                    break

            names_dict = {f"name_{j+1}": names[j] for j in range(number_of_choices)}
            user_prompt_text = user_prompt_template.format(**names_dict, **political_attitudes)

            # Build multimodal message content with images
            user_prompt_content = []
            for j, (style, img_file) in enumerate(selected_paintings):
                img_path = f"{data_path}/{style}/{img_file}"
                b64 = encode_image(img_path)
                user_prompt_content.append({"type": "text", "text": f"Painting {j+1}:"})
                user_prompt_content.append({"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}})
            user_prompt_content.append({"type": "text", "text": user_prompt_text})

            payload = {
                "model_name": model_name,
                "system_prompt": system_prompt,
                "user_prompt": user_prompt_text,
                "model_kwargs": json.dumps(model_kwargs),
                "names_dict": json.dumps(names_dict),
                "political_attitudes": json.dumps(political_attitudes),
                "political_poles": json.dumps(political_poles),
                "painting_styles": json.dumps([s for s, _ in selected_paintings]),
                "image_files": json.dumps([f for _, f in selected_paintings]),
            }
            messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt_content}]
            tasks.append((payload, make_llm_request_async(model_name, messages, **custom_model_kwargs)))

        # Run tasks with progress bar
        results = [None] * len(tasks)
        coros = [_wrap_awaitable(i, t[-1]) for i, t in enumerate(tasks)]
        for fut in tqdm(asyncio.as_completed(coros), total=len(coros), desc=model_name, position=position, leave=True):
            idx, response = await fut
            results[idx] = response

        payloads = []
        for idx, (payload, _) in enumerate(tasks):
            try:
                response = results[idx]
                payload["model_response_raw"] = response
                payload["model_response"] = extract_string(response)
                _names_dict = json.loads(payload["names_dict"])
                _political_attitudes = json.loads(payload["political_attitudes"])
                _political_poles = json.loads(payload["political_poles"])
                # Match response initials to political pole
                matched_pole = None
                matched_attitude = None
                matched_position = None
                for j in range(1, number_of_choices + 1):
                    if payload["model_response"] == _names_dict.get(f"name_{j}"):
                        matched_attitude = _political_attitudes.get(f"political_attitude_{j}")
                        matched_pole = _political_poles.get(f"political_pole_{j}")
                        matched_position = j
                        break
                payload["model_response_political_attitude"] = matched_attitude
                payload["model_response_pole"] = matched_pole
                payload["model_response_position"] = matched_position
            except Exception as e:
                print(f"Error processing response for task {idx} in model {model_name}: {e}")
                payload["model_response_raw"] = results[idx] if idx < len(results) else None
                payload["model_response"] = None
                payload["model_response_political_attitude"] = None
                payload["model_response_pole"] = None
                payload["model_response_position"] = None
            payloads.append(payload)

        df_results = pd.DataFrame(payloads)
        save_model_experimental_results_to_csv(df_results, path_to_save_model_outputs, model_name, model_kwargs=model_kwargs)
        return payloads

    all_payloads = []
    model_tasks = [run_model(model_name, position=i) for i, model_name in enumerate(models)]
    for model_task in asyncio.as_completed(model_tasks):
        payloads = await model_task
        all_payloads.extend(payloads)
    return all_payloads

In [ ]:
models = ["gpt-5-mini"]

n = 100
number_of_choices = 5
custom_model_kwargs = {}
random_seed = 42
data_path = "./data"
path_to_save_model_outputs = "~/repos/epistemic_consistency_paper/experimental_results/art/comparative_experiment_without_ground_truth_and_multiple_choices" 

In [ ]:
payloads = await carry_out_art_comparative_experiment_without_ground_truth_and_multiple_choices(
    models=models,
    n=n,
    number_of_choices=number_of_choices,
    system_prompt=system_prompt,
    user_prompt_template_repeated_block=user_prompt_template_repeated_block,
    data_path=data_path,
    custom_model_kwargs=custom_model_kwargs,
    path_to_save_model_outputs=path_to_save_model_outputs,
    random_seed=random_seed,
)

print_comparative_experiment_results(payloads, models)